<a href="https://colab.research.google.com/github/mohmmadhadi/Algorithms-for-Massive-Data/blob/main/AMD_Mohammadhadi_Shahhosseini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install kaggle

In [2]:
!pip install pyspark spark-nlp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 743.3/743.3 kB 8.8 MB/s eta 0:00:00


##Libraries

In [3]:
import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

import sparknlp
from sparknlp.base import DocumentAssembler, Finisher
from sparknlp.annotator import Tokenizer, Normalizer, StopWordsCleaner, LemmatizerModel

from pyspark.ml import Pipeline
from pyspark.ml.feature import NGram, HashingTF, MinHashLSH

## API & Dataset Downloading

In [4]:
os.environ['KAGGLE_USERNAME'] = "XXXXXXXXXXX"
os.environ['KAGGLE_KEY'] = "XXXXXXXXXXX"

In [5]:
!kaggle datasets download -d mohamedbakhet/amazon-books-reviews

Dataset URL: https://www.kaggle.com/datasets/mohamedbakhet/amazon-books-reviews
License(s): CC0-1.0
100% 1.06G/1.06G [00:09<00:00, 259MB/s]
100% 1.06G/1.06G [00:09<00:00, 117MB/s]


In [6]:
!unzip amazon-books-reviews.zip

Archive:  amazon-books-reviews.zip
  inflating: Books_rating.csv        
  inflating: books_data.csv          


# Spark Session

In [7]:
spark = sparknlp.start()

In [8]:
reviews_df = spark.read.csv("/content/Books_rating.csv", header=True, inferSchema=True)

In [9]:
reviews_df.show(10)

+----------+--------------------+-----+--------------+--------------------+------------------+------------+-----------+--------------------+--------------------+
|        Id|               Title|Price|       User_id|         profileName|review/helpfulness|review/score|review/time|      review/summary|         review/text|
+----------+--------------------+-----+--------------+--------------------+------------------+------------+-----------+--------------------+--------------------+
|1882931173|Its Only Art If I...| NULL| AVCGYZL8FQQTD|"Jim of Oz ""jim-...|               7/7|         4.0|  940636800|Nice collection o...|This is only for ...|
|0826414346|Dr. Seuss: Americ...| NULL|A30TK6U7DNS82R|       Kevin Killian|             10/10|         5.0| 1095724800|   Really Enjoyed It|I don't care much...|
|0826414346|Dr. Seuss: Americ...| NULL|A3UH4UZ4RSVO82|        John Granger|             10/11|         5.0| 1078790400|Essential for eve...|"If people become...|
|0826414346|Dr. Seuss: Ameri

In [10]:
reviews_df.printSchema()

root
 |-- Id: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Price: string (nullable = true)
 |-- User_id: string (nullable = true)
 |-- profileName: string (nullable = true)
 |-- review/helpfulness: string (nullable = true)
 |-- review/score: string (nullable = true)
 |-- review/time: string (nullable = true)
 |-- review/summary: string (nullable = true)
 |-- review/text: string (nullable = true)



In [11]:
row_count = reviews_df.count()
print(f'number of rows:{row_count}')

number of rows:3000000


In [12]:
reviews_df.describe().show()

+-------+--------------------+--------------------+--------------------+-------------------+-----------+-------------------+------------------+--------------------+--------------------+--------------------+
|summary|                  Id|               Title|               Price|            User_id|profileName| review/helpfulness|      review/score|         review/time|      review/summary|         review/text|
+-------+--------------------+--------------------+--------------------+-------------------+-----------+-------------------+------------------+--------------------+--------------------+--------------------+
|  count|             3000000|             2999792|              482421|            2437750|    2437800|            2999633|           2999870|             2999973|             2999935|             2999957|
|   mean|1.0568515696607149E9|   2012.796651763537|  21.767951161877054|  18.29299003322259|        NaN|3.285048033703448E8| 1656.860421970827|1.1270533345949814E9|        

## Preprocessing

As pre-processing, we are going to select columns to use, drop duplicate rows to avoid any over similarity, and check the possibility of removing null values.

We selected id, User_id because of their uniqueness index property and review/text since this column would be the one we focus on calculating the similarity.

In [13]:
null_counts = reviews_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in reviews_df.columns
])

null_counts.show()

+---+-----+-------+-------+-----------+------------------+------------+-----------+--------------+-----------+
| Id|Title|  Price|User_id|profileName|review/helpfulness|review/score|review/time|review/summary|review/text|
+---+-----+-------+-------+-----------+------------------+------------+-----------+--------------+-----------+
|  0|  208|2517579| 562250|     562200|               367|         130|         27|            65|         43|
+---+-----+-------+-------+-----------+------------------+------------+-----------+--------------+-----------+



In [14]:
# dropping duplicates
reviews_df = reviews_df.select("Id", "Title", "review/text").dropDuplicates()

In [15]:
# length of dataframe after removing duplicate rows
reviews_df.count()

2964827

In [16]:
# removing null values
reviews_df_clean = reviews_df.na.drop()

In [17]:
# length of dataframe after cleaning nulls
reviews_df_clean.count()

2964591

In [18]:
#changing review/text column name for easier use
df = reviews_df_clean.withColumnRenamed("review/text", "text")

In [19]:
df = (
    df
    .withColumn("text", F.lower(F.col("text")))
    .withColumn("text", F.regexp_replace(F.col("text"), r"[^\p{L}\p{Nd}\s]+", " "))
    .withColumn("text", F.regexp_replace(F.col("text"), r"\s+", " "))
    .withColumn("text_len", F.length(F.col("text")))
    .filter(F.col("text_len") >= 5)
    .drop("text_len")
    .dropDuplicates(["text"])
)

In [20]:
df.count()

2058299

#### Config: Sample or The Whole Dataset

In [21]:
def nlp_clean(df,
              min_tokens = 3,
              use_sample = True,
              sample_mode = "rows",
              sample_size = 15000,
              sample_fraction = 0.05,
              sample_seed = 42,
              extra_stopwords= None):

  print("=== Spark NLP Cleaning ===")
  if use_sample:
    if sample_mode == "rows":
      df_work = df.orderBy(F.rand(sample_seed)).limit(sample_size)
      print(f"[INFO] Using random sample of {sample_size} rows")
    elif sample_mode == "fraction":
      df_work = df.sample(withReplacement=False, fraction=sample_fraction, seed=sample_seed)
      print(f"[INFO] Using random sample of {sample_fraction*100:.1f}% of data")
    else:
      raise ValueError("sample_mode must be 'rows' or 'fraction'")
  else:
    df_work = df
    print("[INFO] Using full dataset")



  document_assembler = DocumentAssembler().setInputCol("text").setOutputCol("document")

  tokenizer = Tokenizer().setInputCols("document").setOutputCol("token")

  normalizer = Normalizer().setInputCols("token").setOutputCol("normalized").setLowercase(True).setCleanupPatterns(["[^A-Za-z0-9]"])

  stopwords_cleaner = StopWordsCleaner().setInputCols("normalized").setOutputCol("cleanTokens").setCaseSensitive(False)

  lemmatizer = LemmatizerModel.pretrained("lemma_antbnc", "en").setInputCols("cleanTokens").setOutputCol("lemma")

  finisher = Finisher().setInputCols(["lemma"]).setOutputCols(["final_tokens"]).setCleanAnnotations(True)

  pipe = Pipeline(stages=[document_assembler, tokenizer, normalizer, stopwords_cleaner, lemmatizer, finisher])

  model = pipe.fit(df_work)
  result = model.transform(df_work)

  result = (result.withColumn("tok_size", F.size("final_tokens")).filter(F.col("tok_size") >= min_tokens).select("Id", "text", "final_tokens"))

  # Keeping the dataframe in memory and store the results in cache
  result = result.cache()
  _ = result.count()
  return result

In [22]:
df_tok = nlp_clean(df, use_sample=True, sample_mode="rows")

=== Spark NLP Cleaning ===
[INFO] Using random sample of 15000 rows
lemma_antbnc download started this may take some time.
Approximate size to download 907.6 KB
[OK!]


In [23]:
df_tok.show(5)

+----------+--------------------+--------------------+
|        Id|                text|        final_tokens|
+----------+--------------------+--------------------+
|0312033613|1939 is hailed as...|[1939, hail, one,...|
|B0006E2W9M| set during the g...|[set, great, depr...|
|0570053641|nikki rach writes...|[nikki, rach, wri...|
|1562533703|the book is very ...|[book, helpful, q...|
|0613066421|this was the thir...|[third, binchy, n...|
+----------+--------------------+--------------------+
only showing top 5 rows



## Making Shingles

In [24]:
ngram = NGram(n=3, inputCol = "final_tokens", outputCol = "Shingles")
df_shingles = ngram.transform(df_tok)

In [25]:
df_shingles.show(5)

+----------+--------------------+--------------------+--------------------+
|        Id|                text|        final_tokens|            Shingles|
+----------+--------------------+--------------------+--------------------+
|0312033613|1939 is hailed as...|[1939, hail, one,...|[1939 hail one, h...|
|B0006E2W9M| set during the g...|[set, great, depr...|[set great depres...|
|0570053641|nikki rach writes...|[nikki, rach, wri...|[nikki rach write...|
|1562533703|the book is very ...|[book, helpful, q...|[book helpful que...|
|0613066421|this was the thir...|[third, binchy, n...|[third binchy nov...|
+----------+--------------------+--------------------+--------------------+
only showing top 5 rows



## Hashing

In [26]:
htf = HashingTF(inputCol="Shingles", outputCol="features", numFeatures=1<<18, binary=True)
df_hashed = htf.transform(df_shingles).persist()

In [27]:
df_hashed.show(5)

+----------+--------------------+--------------------+--------------------+--------------------+
|        Id|                text|        final_tokens|            Shingles|            features|
+----------+--------------------+--------------------+--------------------+--------------------+
|0312033613|1939 is hailed as...|[1939, hail, one,...|[1939 hail one, h...|(262144,[331,1531...|
|B0006E2W9M| set during the g...|[set, great, depr...|[set great depres...|(262144,[4298,908...|
|0570053641|nikki rach writes...|[nikki, rach, wri...|[nikki rach write...|(262144,[40526,44...|
|1562533703|the book is very ...|[book, helpful, q...|[book helpful que...|(262144,[2277,559...|
|0613066421|this was the thir...|[third, binchy, n...|[third binchy nov...|(262144,[3902,542...|
+----------+--------------------+--------------------+--------------------+--------------------+
only showing top 5 rows



## MinHash and Jaccard

Config

In [28]:
SIM_THRESHOLD = 0.1
NUM_HASH_TABLES = 6

#### MinHash

In [29]:
lsh = MinHashLSH(inputCol="features", outputCol="hashes", numHashTables=NUM_HASH_TABLES)
lsh_model = lsh.fit(df_hashed)

#### Jaccard

In [30]:
pairs = (
    lsh_model.approxSimilarityJoin(df_hashed, df_hashed, 1.0 - SIM_THRESHOLD, distCol="jaccard_dist")
    .filter(F.col("datasetA.Id") < F.col("datasetB.Id"))   # drop (a,a) and dup order
    .select(
        F.col("datasetA.Id").alias("id_a"),
        F.col("datasetB.Id").alias("id_b"),
        (1.0 - F.col("jaccard_dist")).alias("jaccard_sim"),
        F.col("datasetA.text").alias("text_a"),
        F.col("datasetB.text").alias("text_b"),
    )
    .orderBy(F.desc("jaccard_sim"))
)


In [48]:
pairs.show(20, truncate=100)
print("Pairs ≥ threshold:", pairs.count())

+----------+----------+-------------------+----------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------+
|      id_a|      id_b|        jaccard_sim|                                                                                              text_a|                                                                                              text_b|
+----------+----------+-------------------+----------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------+
|061858532X|B00005UVH9| 0.2857142857142857|    this is the best book i ve read in a long time it s amazing i ll never forget this book read it |    one of the best books i ve read in a very long time the characters will stay with you for years |
|0553471058|0889

## Before De-Duplication of text column separately

In [70]:
pairs.show(20, truncate=100)
print("Pairs ≥ threshold:", pairs.count())

+----------+----------+-----------+----------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------+
|      id_a|      id_b|jaccard_sim|                                                                                              text_a|                                                                                              text_b|
+----------+----------+-----------+----------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------+
|B000GQK706|B000PIIMPW|        1.0|this book is a disapointing buy first of all it is not leather bound but made with cheap artifici...|this book is a disapointing buy first of all it is not leather bound but made with cheap artifici...|
|B0008BL2SA|B0008CXTHG|        1.0|little women 

In [72]:
reviews_df_clean.groupBy("review/text").count().orderBy(F.desc("count")).show(20, truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|review/text                                                                                                                                                                                                                                                                                                                                         

In [73]:
df.groupBy("text").count().orderBy(F.desc("count")).show(20, truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|text                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   |count|
+-------